In [1]:
import os
import numpy as np
import pickle

In [2]:

def load(name):
    k = 1
    while os.path.isfile(f"{name}_{k}.pkl"):
        k+=1
    k-=1
    with open(f'{name}_{k}.pkl','rb') as f:
        contentbis = pickle.load(f)
    print(f'{name}_{k}.pkl')

    return contentbis


N = 3000
folder = 'non_exclusive_axis_exploration3' 
name = f'{folder}/rand_run_{N}'
content_rand = load(name)

non_exclusive_axis_exploration3/rand_run_3000_0.pkl


In [3]:
content_rand['reward']

array([[  0.        ,   0.        ,   0.        , ...,   0.        ,
          0.        ,   0.        ],
       [ 55.84704367,  55.84704367,  55.84704367, ...,  55.84704367,
         55.84704367,  55.84704367],
       [ 21.79398185,  34.05306182,  34.05306182, ...,  21.79398185,
         34.05306182,  34.05306182],
       ...,
       [ 42.23201896, 270.68680375, 248.8928219 , ..., 304.73986557,
        304.73986557, 304.73986557],
       [198.21320006,  98.33103211, 544.57572745, ..., 544.57572745,
        510.52266563, 544.57572745],
       [156.95070626, 160.03586224, 454.74458275, ..., 454.74458275,
        476.53856461, 510.59162642]], shape=(3000, 159))

In [4]:
def content_o_alp_space(content):
    NN,F = content['tabular_view'].shape
    return np.concatenate((content['tabular_view'].reshape(N,1,F),content['reward'].reshape(N,1,F)),axis=1)

In [5]:
o_alp_space = content_o_alp_space(content_rand)
print(o_alp_space.shape)

(3000, 2, 159)


In [6]:
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import GridSearchCV
import time

In [7]:
def gmm_bic_score(estimator, X):
    """Callable to pass to GridSearchCV that will use the BIC score."""
    # Make it negative since GridSearchCV expects a score to maximize 
    # and one wants to minize bic score
    return -estimator.bic(X)

# One instantiate the gmm class and grid search

In [8]:
param_grid = {
    "n_components": range(2, 7),
    "covariance_type": ["diag", "full"],
}
grid_search = GridSearchCV(
    GaussianMixture(), param_grid=param_grid, scoring=gmm_bic_score
)

In [9]:
X = o_alp_space[-250:,:,0]

In [10]:
start = time.time()
grid_search.fit(X)
print(time.time() - start)

0.5202620029449463


In [16]:
import time

In [17]:
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import ParameterGrid
from joblib import Parallel, delayed
import numpy as np

def fit_gmm_for_dataset(dataset, params):
    """Fit GMM for a single dataset with given parameters and return the model"""
    gmm = GaussianMixture(**params)
    gmm.fit(dataset)
    return {
        'params': params,
        'bic': gmm.bic(dataset),
        'aic': gmm.aic(dataset),
        'converged': gmm.converged_,
        'model': gmm  # Return the fitted model!
    }

def parallel_gmm_search_with_models(datasets, param_grid, n_jobs=-1):
    """Perform parallel GMM grid search and return best models"""
    
    best_models = []
    all_results = []
    
    for i, dataset in enumerate(datasets):
        if i%50==0:
            print(f"Processing dataset {i+1}/{len(datasets)}")
        
        param_combinations = list(ParameterGrid(param_grid))
        
        # Parallel execution
        results = Parallel(n_jobs=n_jobs)(
            delayed(fit_gmm_for_dataset)(dataset, params)
            for params in param_combinations
        )
        
        # Find best model based on BIC (lower is better)
        best_result = min(results, key=lambda x: x['bic'])
        
        # Store the best model and its info
        best_models.append({
            'dataset_idx': i,
            'model': best_result['model'],  # The fitted GMM model
            'best_params': best_result['params'],
            'best_bic': best_result['bic'],
            'best_aic': best_result['aic']
        })
        
        all_results.append(results)
    
    return best_models, all_results

In [26]:
# Perform grid search and get best models
datasets = [np.random.randn(1000, 2) for _ in range(3)]
param_grid = {
    'n_components': [1,2, 3, 4],
    'covariance_type': ['full', 'diag'],
    'random_state': [42]
}
start_time = time.time()
best_models, all_results = parallel_gmm_search_with_models(datasets, param_grid, n_jobs=4)
print('end time',time.time() - start_time)
# Now use the best models for sampling
samples_per_dataset = 1
all_generated_samples = []

for i, best_model_info in enumerate(best_models):
    model = best_model_info['model']
    
    print(f"\n--- Best Model for Dataset {i} ---")
    print(f"Parameters: {best_model_info['best_params']}")
    print(f"BIC: {best_model_info['best_bic']:.2f}")
    print(f"Number of components: {model.n_components}")
    
    # Generate samples from the best distribution
    generated_samples, component_labels = model.sample(samples_per_dataset)
    all_generated_samples.append(generated_samples)
    
    print(f"Generated {len(generated_samples)} samples from dataset {i}")
    print(f"Sample mean: {np.mean(generated_samples, axis=0)}")
    print(f"Sample shape: {generated_samples.shape}")

# all_generated_samples now contains synthetic data from each best distribution

Processing dataset 1/3
end time 0.20218777656555176

--- Best Model for Dataset 0 ---
Parameters: {'covariance_type': 'diag', 'n_components': 1, 'random_state': 42}
BIC: 5724.18
Number of components: 1
Generated 1 samples from dataset 0
Sample mean: [ 0.52297612 -0.11807952]
Sample shape: (1, 2)

--- Best Model for Dataset 1 ---
Parameters: {'covariance_type': 'diag', 'n_components': 1, 'random_state': 42}
BIC: 5749.99
Number of components: 1
Generated 1 samples from dataset 1
Sample mean: [ 0.4901035 -0.1309702]
Sample shape: (1, 2)

--- Best Model for Dataset 2 ---
Parameters: {'covariance_type': 'diag', 'n_components': 1, 'random_state': 42}
BIC: 5679.61
Number of components: 1
Generated 1 samples from dataset 2
Sample mean: [ 0.51646314 -0.10264606]
Sample shape: (1, 2)


In [27]:
import dask
from dask.distributed import Client
import dask.bag as db

def setup_dask_cluster():
    """Set up a local Dask cluster"""
    client = Client(n_workers=4, threads_per_worker=1)
    return client

def dask_parallel_gmm(datasets, param_grid):
    """Using Dask for parallel processing"""
    client = setup_dask_cluster()
    
    param_combinations = list(ParameterGrid(param_grid))
    
    # Create lazy computations
    lazy_results = []
    for i, dataset in enumerate(datasets):
        for params in param_combinations:
            # Create lazy task
            result = dask.delayed(fit_gmm_for_dataset)(dataset, params)
            lazy_results.append((i, result))
    
    # Compute in parallel
    computed_results = dask.compute(*[r[1] for r in lazy_results])
    
    # Organize results
    final_results = {}
    for (dataset_idx, _), result in zip(lazy_results, computed_results):
        if dataset_idx not in final_results:
            final_results[dataset_idx] = []
        final_results[dataset_idx].append(result)
    
    return final_results

In [16]:
start = time.time()
results = dask_parallel_gmm(datasets, param_grid)
print(time.time()-start)

5.33003568649292
